In [0]:
# HIGH_GARDEN_CATALOG_PARAMETER
dbutils.widgets.text("catalog", "high_garden")
catalog = dbutils.widgets.get("catalog").strip() or "high_garden"
print(f"Using Unity Catalog: {catalog}")


In [0]:
from pyspark.sql import functions as F
import re

SOURCE_TABLE = f"{catalog}.bronze.coffee_raw"
TARGET_TABLE = f"{catalog}.silver.coffee_consumption"

bronze_df = spark.table(SOURCE_TABLE)

In [0]:
bronze_df.printSchema()

print("Rows:", bronze_df.count())
print("Columns:", len(bronze_df.columns))

In [0]:
year_columns = [
    column
    for column in bronze_df.columns
    if re.fullmatch(r"\d{4}_\d{2}", column)
]

assert len(year_columns) == 30

In [0]:
stack_values = ", ".join(
    [
        f"'{column.replace('_', '/')}', `{column}`"
        for column in year_columns
    ]
)

stack_expression = (
    f"stack({len(year_columns)}, {stack_values}) "
    "as (crop_year, domestic_consumption)"
)

silver_df = bronze_df.selectExpr(
    "country",
    "coffee_type",
    stack_expression
)

In [0]:
silver_df = (
    silver_df
    .withColumn(
        "start_year",
        F.substring("crop_year", 1, 4).cast("int")
    )
    .withColumn(
        "end_year",
        F.col("start_year") + 1
    )
    .withColumn(
        "domestic_consumption",
        F.col("domestic_consumption").cast("double")
    )
)

In [0]:
silver_df = silver_df.withColumn(
    "zero_flag",
    (
        F.col("domestic_consumption") == 0
    ).cast("int")
)

In [0]:
silver_df = silver_df.select(
    "country",
    "coffee_type",
    "crop_year",
    "start_year",
    "end_year",
    "domestic_consumption",
    "zero_flag"
)

In [0]:
assert silver_df.count() == 1650

assert (
    silver_df
    .select("country")
    .distinct()
    .count()
    == 55
)

assert (
    silver_df
    .select("crop_year")
    .distinct()
    .count()
    == 30
)

assert (
    silver_df
    .filter(
        F.col("domestic_consumption") < 0
    )
    .count()
    == 0
)

In [0]:
duplicates_df = (
    silver_df
    .groupBy(
        "country",
        "crop_year"
    )
    .count()
    .filter(
        F.col("count") > 1
    )
)

assert duplicates_df.count() == 0

In [0]:
null_consumption = (
    silver_df
    .filter(
        F.col("domestic_consumption").isNull()
    )
    .count()
)

assert null_consumption == 0

In [0]:
zero_count = (
    silver_df
    .filter(
        F.col("domestic_consumption") == 0
    )
    .count()
)

print("Zero-valued observations:", zero_count)

In [0]:
(
    silver_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(TARGET_TABLE)
)

In [0]:
display(spark.sql(f"""
SELECT *
FROM {catalog}.silver.coffee_consumption
ORDER BY country, start_year
LIMIT 30;
"""))


In [0]:
display(spark.sql(f"""
SELECT
    COUNT(*) AS rows,
    COUNT(DISTINCT country) AS countries,
    COUNT(DISTINCT crop_year) AS periods,
    SUM(zero_flag) AS zero_observations
FROM {catalog}.silver.coffee_consumption;
"""))
